In [10]:
import os
import re

def searchFile(ROOT_PATH,saveFile):
    if not os.path.isdir(os.path.dirname(saveFile)):
        os.makedirs(os.path.dirname(saveFile))
    with open(saveFile,"w",encoding="utf-8")as f:
        for pathname,dirnames,filenames in os.walk(ROOT_PATH):
            for filename in filenames:
                if filename.endswith(".java"):
                    fullPath=os.path.join(pathname,filename)
                    if "test" not in fullPath and "info" not in fullPath and "example" not in fullPath:
                        print(fullPath,file=f)
                        
gitProject="C:\\Users\\sugii syuji\\jsoup"
savaFile="C:\\Users\\sugii syuji\\SpoonCKv2\\experiment.txt"
searchFile(gitProject,savaFile)

#gitプロジェクトのパスを渡して、gitプロジェクト内に含まれる test info を除いたjavaファイルの一覧を取得する

In [ ]:
import subprocess
import os
import shutil


def getLog(rootProject,LogDir):
    gitProject = rootProject
    saveLogDir = LogDir

    if os.path.isdir(saveLogDir):
        shutil.rmtree(saveLogDir)
    os.makedirs(saveLogDir)

    beLatestVersion = "git checkout master"
    subprocess.run(beLatestVersion.split(), cwd=gitProject, check=True)

    getLog = "git log --pretty=format:%H --name-only --diff-filter=ACMR -- *.java"
    hash = subprocess.run(
        getLog.split(), cwd=gitProject, capture_output=True, check=True, text=True
    )
    blocks = hash.stdout.split("\n\n")
    k = 0
    count=0
    for commit in blocks:
        commitLines = commit.split("\n")
        filterList = [i for i in commitLines if "test" not in i and "info" not in i and "example" not in i]
        if len(filterList) >= 2:
            count=count+len(filterList)-1
            k = k + 1
            for i in range(1, len(filterList)):
                filterList[i] = os.path.join(gitProject , filterList[i])
            fileWrite = "\n".join(filterList)
            with open(os.path.join(saveLogDir,"logHash.txt"),"a",encoding="utf-8") as f:
                f.write(str(k)+","+filterList[0]+"\n")
            with open(os.path.join(saveLogDir , str(k) + ".txt"), "w", encoding="utf-8") as f:
                f.write(fileWrite)
getLog("C:\\Users\\syuuj\\gitProject\\jitwatch","C:\\Users\\syuuj\\SpoonCKv2\\logData")       
# gitプロジェクトのパスから、test info を除いた Javaファイル を変更したlog( commitID と　変更Javaファイル )をlogDataに保存する　※１つのコミットに対して１つのファイル
#git show

In [38]:
import os
import subprocess
import shutil
def getBugIssue(gitProject,saveFile):
    if os.path.isdir(os.path.dirname(saveFile)):
        shutil.rmtree(os.path.dirname(saveFile))
    os.makedirs(os.path.dirname(saveFile))
    gitCheckout="git checkout master"
    subprocess.run(gitCheckout.split(),cwd=gitProject)
    getFixLog=["gh","issue","list","-l","bug","--limit","9999","--state","closed","--json","url","--jq",".[].url"]
    bugNum=subprocess.run(getFixLog,cwd=gitProject,capture_output=True,text=True)
    with open(saveFile,"w",encoding="utf-8") as f:
        f.write(bugNum.stdout)
    
getBugIssue("C:\\Users\\syuuj\\gitProject\\jsoup","C:\\Users\\syuuj\\bugData\\jsoup\\bug.txt")       

In [39]:
import os
import subprocess
def getBugLog(gitProject,issueNumFile,saveFile):
    if os.path.isfile(saveFile):
        os.remove(saveFile)
    with open(issueNumFile,"r",encoding="utf-8") as f:
        issueNum=f.read()
    issueNumList=issueNum.split("\n")
    countIssue=0
    countBugFix=0
    for num in issueNumList:
        countIssue=countIssue+1
        if num=="":
            continue
        #getID=["git","log","--pretty=format:%H","--extended-regexp",'--grep=#'+num+'([^0-9]|$)']
        getID=["git","log","--pretty=format:%H","--extended-regexp",'--grep='+num]
        tmp=subprocess.run(getID,cwd=gitProject,capture_output=True,text=True)
        if len(tmp.stdout)==0:
            continue
        countBugFix=countBugFix+1
        with open(saveFile,"a",encoding="utf-8") as f:
            f.write(num+"\n")
            f.write(tmp.stdout+"\n\n")
    print(gitProject+" : "+str(countIssue)+"(Issue数) "+str(countBugFix)+"(紐づいてるIssue数)")
        
getBugLog("C:\\Users\\syuuj\\gitProject\\jsoup","C:\\Users\\syuuj\\bugData\\jsoup\\bug.txt","C:\\Users\\syuuj\\bugData\\jsoup\\bugLogData.txt")       

C:\Users\syuuj\gitProject\jsoup : 184(Issue数) 0(紐づいてるIssue数)


In [5]:
import os
import subprocess
import shutil
def getBugIssue(gitProject,saveFile):
    if os.path.isdir(os.path.dirname(saveFile)):
        shutil.rmtree(os.path.dirname(saveFile))
    os.makedirs(os.path.dirname(saveFile))

    getFixLog=["gh","issue","list","-l","bug","--limit","9999","--state","closed","--json","number","--jq",".[].number"]
    #getFixLog=["gh","issue","list","-l","type=defect","--limit","9999","--state","closed","--json","url","--jq",".[].url"]

    bugNum=subprocess.run(getFixLog,cwd=gitProject,capture_output=True,text=True)
    with open(saveFile,"w",encoding="utf-8") as f:
        f.write(bugNum.stdout)

def getBugLog(gitProject,issueNumFile,saveFile):
    if os.path.isfile(saveFile):
        os.remove(saveFile)
    with open(issueNumFile,"r",encoding="utf-8") as f:
        issueNum=f.read()
    issueNumList=issueNum.split("\n")
    countIssue=0
    countBugFix=0
    for num in issueNumList:
        countIssue=countIssue+1
        if num=="":
            continue
        getID=["git","log","--pretty=format:%H","--extended-regexp",'--grep=#'+num+'([^0-9]|$)']
        #getID=["git","log","--pretty=format:%H","--extended-regexp",'--grep='+num]
        tmp=subprocess.run(getID,cwd=gitProject,capture_output=True,text=True)
        if len(tmp.stdout)==0:
            continue
        countBugFix=countBugFix+1
        with open(saveFile,"a",encoding="utf-8") as f:
            f.write(num+"\n")
            f.write(tmp.stdout+"\n\n")
    print(gitProject+" : "+str(countIssue)+"(Issue数) "+str(countBugFix)+"(紐づいてるIssue数)")

def allDo(gitProject,saveIssueFile,saveBufLogFile):
    getBugIssue(gitProject,saveIssueFile)
    getBugLog(gitProject,saveIssueFile,saveBufLogFile)

#tmp=r"c:\Users\syuuj\gitProject\mybatis-3 c:\Users\syuuj\gitProject\nanohttpd c:\Users\syuuj\gitProject\netty c:\Users\syuuj\gitProject\redisson c:\Users\syuuj\gitProject\retrofit c:\Users\syuuj\gitProject\shopizer c:\Users\syuuj\gitProject\vert.x c:\Users\syuuj\gitProject\webmagic c:\Users\syuuj\gitProject\ysoserial c:\Users\syuuj\gitProject\zookeeper c:\Users\syuuj\gitProject\checkstyle c:\Users\syuuj\gitProject\CoreNLP c:\Users\syuuj\gitProject\dbeaver c:\Users\syuuj\gitProject\fastjson c:\Users\syuuj\gitProject\gson c:\Users\syuuj\gitProject\guava c:\Users\syuuj\gitProject\HikariCP c:\Users\syuuj\gitProject\jedis c:\Users\syuuj\gitProject\jenkins c:\Users\syuuj\gitProject\jitwatch c:\Users\syuuj\gitProject\jsoup c:\Users\syuuj\gitProject\junit4 c:\Users\syuuj\gitProject\libgdx c:\Users\syuuj\gitProject\mapstruct c:\Users\syuuj\gitProject\mockserver"
#gitProjects=tmp.split(" ")
gitProjects=["c:\\Users\\syuuj\\gitProject\\guava"]
for gitProject in gitProjects:
    name=os.path.basename(gitProject)
    bugText=os.path.join("C:\\Users\\syuuj\\bugData",name,"bug.txt")
    bugLogData=os.path.join("C:\\Users\\syuuj\\bugData",name,"bugLogData.txt")
    allDo(gitProject,bugText,bugLogData)
    

c:\Users\syuuj\gitProject\guava : 1(Issue数) 0(紐づいてるIssue数)


In [12]:
import os 
import subprocess

def ExcuteSpoon(full,diff,tmpResult):
    outPutDir=tmpResult    #結果を保存するフォルダ
    SpoonRootDir=os.path.join(os.getcwd(),"spoon")
    spoonCMD=["java","-jar","target/demo-1.0-snapshot.jar",full,diff,outPutDir]
    error=subprocess.run(spoonCMD,cwd=SpoonRootDir,capture_output=True,text=True)
    print(error.stderr)
    
ExcuteSpoon("C:\\Users\\sugii syuji\\SpoonCKv2\\experiment.txt","C:\\Users\\sugii syuji\\SpoonCKv2\\experiment.txt","C:\\Users\\sugii syuji\\SpoonCKv2\\tmpresult\\2")

SLF4J(W): No SLF4J providers were found.
SLF4J(W): Defaulting to no-operation (NOP) logger implementation
SLF4J(W): See https://www.slf4j.org/codes.html#noProviders for further details.

